# Notebook 03 — CNN for Sequence Classification

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 3 of 12  
**Time estimate:** 90 minutes

> A convolutional filter scanning a DNA sequence is a learned Position Weight Matrix.
> This is the DeepBind insight (Alipanahi 2015) — and understanding it lets you
> interpret what every sequence CNN has actually learned.

---
## Step 1 — Motivation

A PWM scores a fixed motif at every position, takes the max, and classifies.
A CNN does the same thing — but it learns the motif from data, learns multiple
motifs simultaneously, and can combine motif co-occurrences in deeper layers.
DeepBind outperformed PWMs on 506 ChIP-seq datasets. Understanding the architecture
lets you apply it to any sequence classification problem in genomics.

---
## Step 2 — Intuition

**1D convolution for sequences:**
- Input: one-hot encoded sequence, shape `(4, L)` — 4 nucleotides × L positions
- Convolutional filter: shape `(n_filters, 4, kernel_size)` — n_filters learned PWMs,
  each of width `kernel_size`
- Output of one filter at position $i$: inner product of filter and the `(4 × kernel_size)`
  window starting at position $i$ → a scalar score
- After sliding: a 1D activation map of length $L - \text{kernel\_size} + 1$

**Max pooling (global):**
Take the maximum activation across all positions → "does this motif appear anywhere?"
This is analogous to the max PWM score from Module 13 NB13.

**Multiple filters:**
Each filter learns a different motif. After global max pooling, you have a feature
vector of size `n_filters` — one score per learned motif. The FC layer then combines
these motif scores to make the final prediction.

**Deeper CNNs:**
Second conv layer operates on the outputs of the first — learns combinations of motifs
(co-occurrence patterns). Enformer uses 6 convolutional layers with residual connections
before the transformer on a 196kb window.

---
## Step 3 — Biological Background

**DeepBind (Alipanahi 2015, Nature Biotechnology):**
First large-scale CNN application to TF binding.
- Input: 101-bp ChIP-seq peak sequences, one-hot encoded
- Architecture: Conv(16 filters, width 24) → ReLU → global max pool → FC(32) → FC(1)
- Trained on positive (ChIP-seq peaks) vs. negative (shuffled/flanking) sequences
- AUROC improvement over PWM: ~15–20% on average across 506 TFs
- Motif visualization: take top-activated sequences for each filter, extract the
  aligned subsequence, compute a PWM — directly comparable to JASPAR

**Enformer (Avsec 2021, Nature Methods):**
Predicts gene expression, histone marks, and transcription across 5,313 genomic tracks
from a 196kb DNA window. Architecture: Conv stack → Transformer encoder → FC per track.
The CNN handles local sequence features; the transformer integrates long-range interactions.

**Why global max pooling?**
TF binding is position-invariant: the motif can appear anywhere in the sequence.
Global max pool captures "does the motif appear" regardless of where.
Average pooling would dilute the signal. This is a domain-knowledge-informed
architectural decision, not arbitrary.

---
## Step 4 — Mathematical Explanation

**1D convolution (for sequences):**
Filter $f$ of shape $(C_{in}, k)$, input $X$ of shape $(C_{in}, L)$:
$$\text{Conv}(X, f)[i] = \sum_{c=0}^{C_{in}-1} \sum_{j=0}^{k-1} X[c, i+j] \cdot f[c, j]$$

Output length: $L_{out} = L - k + 1$ (no padding) or $L$ (same padding).

**Multiple filters:** $F \in \mathbb{R}^{n_{filters} \times C_{in} \times k}$
→ output $H \in \mathbb{R}^{n_{filters} \times L_{out}}$

**Global max pool:**
$$\text{GMP}(H)[f] = \max_{i} H[f, i]$$

**Full DeepBind-style forward pass:**
$$\mathbf{x}_{\text{onehot}} \xrightarrow{\text{Conv + ReLU}} H \xrightarrow{\text{GMP}} \mathbf{h} \xrightarrow{\text{FC + ReLU}} \mathbf{z} \xrightarrow{\text{FC + Sigmoid}} \hat{y}$$

**Receptive field:** How many input positions does one output unit depend on?
After $k$ convolutional layers with kernel size $k_l$:
$\text{RF} = 1 + \sum_{l=1}^{k} (k_l - 1)$
Global max pool makes the effective receptive field the full sequence length.

In [ ]:
# Step 6 — Python: DeepBind-style CNN in PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ---- Sequence dataset (same as Module 13 NB13) ----
NUCL = 'ACGT'
NUCL_IDX = {b: i for i, b in enumerate(NUCL)}

def one_hot_batch(sequences):
    """Convert list of DNA strings to (n, 4, L) tensor."""
    L = len(sequences[0])
    X = np.zeros((len(sequences), 4, L), dtype=np.float32)
    for i, seq in enumerate(sequences):
        for j, b in enumerate(seq):
            if b in NUCL_IDX:
                X[i, NUCL_IDX[b], j] = 1.0
    return X

MOTIF = 'CCGCGNGGNGG'
MOTIF_L = 11
SEQ_L = 50  # longer than Module 13 for more interesting patterns

rng = np.random.default_rng(42)
def generate_seqs(n_pos=600, n_neg=600):
    seqs, labels = [], []
    for _ in range(n_pos):
        seq = list(''.join(rng.choice(list(NUCL), SEQ_L)))
        pos = rng.integers(0, SEQ_L - MOTIF_L)
        for j, b in enumerate(MOTIF):
            if b == 'N':
                seq[pos+j] = rng.choice(list(NUCL))
            elif rng.random() < 0.1:
                seq[pos+j] = rng.choice([x for x in NUCL if x != b])
            else:
                seq[pos+j] = b
        seqs.append(''.join(seq)); labels.append(1)
    for _ in range(n_neg):
        seqs.append(''.join(rng.choice(list(NUCL), SEQ_L))); labels.append(0)
    return seqs, np.array(labels)

sequences, y_labels = generate_seqs()
X_onehot = one_hot_batch(sequences)
print(f'Dataset: {len(sequences)} sequences, shape {X_onehot.shape}')

# ---- PyTorch Dataset ----
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

full_ds = SeqDataset(X_onehot, y_labels)
n_train = int(0.8 * len(full_ds))
train_ds, val_ds = random_split(full_ds, [n_train, len(full_ds)-n_train])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)

# ---- DeepBind-style CNN ----
class DeepBind(nn.Module):
    def __init__(self, n_filters=16, kernel_size=11, fc_units=32):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, kernel_size, padding=0)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)  # global max pool
        self.fc1 = nn.Linear(n_filters, fc_units)
        self.fc2 = nn.Linear(fc_units, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        h = self.pool(self.relu(self.conv(x)))  # (batch, n_filters, 1)
        h = h.squeeze(-1)                         # (batch, n_filters)
        h = self.relu(self.fc1(h))
        return self.sigmoid(self.fc2(h))
    
    def get_filter_activations(self, x):
        """Return pre-pool activations for motif visualization."""
        return self.relu(self.conv(x)).detach().cpu().numpy()

model = DeepBind(n_filters=16, kernel_size=11, fc_units=32).to(device)
print(f'\nModel parameters: {sum(p.numel() for p in model.parameters())}')

# ---- Training loop with AUROC tracking ----
from sklearn.metrics import roc_auc_score

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

train_losses, val_aurocs = [], []
for epoch in range(50):
    model.train()
    bl = []
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        pred = model(X_b.to(device))
        loss = criterion(pred, y_b.to(device))
        loss.backward()
        optimizer.step()
        bl.append(loss.item())
    train_losses.append(np.mean(bl))
    
    model.eval()
    y_probs, y_true = [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            probs = model(X_b.to(device)).cpu().numpy().ravel()
            y_probs.extend(probs); y_true.extend(y_b.numpy().ravel())
    auroc = roc_auc_score(y_true, y_probs)
    val_aurocs.append(auroc)
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1:2d}: loss={train_losses[-1]:.4f}, val AUROC={auroc:.4f}')

# ---- Filter visualization: convert filters to PWMs ----
with torch.no_grad():
    filters = model.conv.weight.cpu().numpy()  # (n_filters, 4, kernel_size)

print(f'\nFilter shapes: {filters.shape}')
print(f'Learned filter 0 (should resemble the planted motif):')
print(np.argmax(filters[0], axis=0))  # argmax per position = consensus nucleotide
consensus_0 = ''.join(NUCL[i] for i in np.argmax(filters[0], axis=0))
print(f'Consensus (filter 0): {consensus_0}')
print(f'Planted motif:        {MOTIF}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

# Panel A: Training curves
ax = axes[0]
ax.plot(train_losses, 'steelblue', lw=1.5, label='Train loss')
ax2 = ax.twinx()
ax2.plot(val_aurocs, 'tomato', lw=1.5, label='Val AUROC')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE loss', color='steelblue')
ax2.set_ylabel('AUROC', color='tomato')
ax.set_title('A. Training curves\n(CNN for TF binding)')
lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=8, loc='center right')

# Panel B: Visualize top 4 learned filters as sequence logos
ax = axes[1]
colors_nucl = {'A': '#00a651', 'C': '#3953a4', 'G': '#fdb813', 'T': '#ed1c24'}
n_show = min(4, filters.shape[0])
for fi in range(n_show):
    flt = filters[fi]  # (4, kernel_size)
    # Normalize to probabilities (softmax for visualization)
    flt_pos = np.exp(flt) / np.exp(flt).sum(axis=0, keepdims=True)
    # Plot at vertical offset
    for pos_j in range(flt.shape[1]):
        bottom = fi * 1.2
        order = np.argsort(flt_pos[:, pos_j])
        for b_idx in order:
            h = flt_pos[b_idx, pos_j] * 1.0
            ax.bar(pos_j, h, bottom=bottom, color=list(colors_nucl.values())[b_idx],
                     width=0.8, edgecolor='none')
ax.set_xlabel('Filter position')
ax.set_ylabel('Filter index')
ax.set_yticks([fi * 1.2 + 0.5 for fi in range(n_show)])
ax.set_yticklabels([f'F{fi}' for fi in range(n_show)], fontsize=8)
ax.set_title('B. Learned filter logos\n(filter 0 should match planted motif)')
from matplotlib.patches import Patch
ax.legend([Patch(color=c, label=b) for b, c in colors_nucl.items()], 
            list(colors_nucl.keys()), fontsize=7, loc='upper right')

# Panel C: Activation map for one positive sequence (position-specific)
ax = axes[2]
model.eval()
pos_seq = sequences[0]  # a positive sequence
X_one = torch.tensor(one_hot_batch([pos_seq]), dtype=torch.float32).to(device)
with torch.no_grad():
    acts = model.get_filter_activations(X_one)[0]  # (n_filters, L-k+1)
im = ax.imshow(acts, cmap='hot', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_xlabel('Sequence position')
ax.set_ylabel('Filter')
ax.set_title(f'C. Filter activations (one + seq)\n(bright = motif match at this position)')

# Panel D: CNN vs. k-mer baseline AUROC comparison
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# k-mer features (same as Module 13 NB13)
def kmer_features(seq, k=4):
    kmers = [''.join(p) for p in product(NUCL, repeat=k)]
    kmer_idx = {km: i for i, km in enumerate(kmers)}
    vec = np.zeros(len(kmers), dtype=np.float32)
    for i in range(len(seq)-k+1):
        km = seq[i:i+k]
        if km in kmer_idx: vec[kmer_idx[km]] += 1
    return vec / max(len(seq)-k+1, 1)

X_kmer = np.array([kmer_features(s, k=4) for s in sequences])
from sklearn.model_selection import cross_val_score, StratifiedKFold
kmer_auc = cross_val_score(LogisticRegression(C=1, max_iter=2000),
                             StandardScaler().fit_transform(X_kmer), y_labels,
                             cv=StratifiedKFold(5), scoring='roc_auc')

ax = axes[3]
ax.bar(['k=4 k-mer\n+ LogReg', 'CNN\n(DeepBind-style)'],
         [kmer_auc.mean(), max(val_aurocs)],
         yerr=[kmer_auc.std(), 0], capsize=4,
         color=['steelblue', 'tomato'], edgecolor='black', alpha=0.85)
ax.set_ylabel('AUROC')
ax.set_title('D. CNN vs. k-mer baseline\n(CNN learns position-aware motifs)')
ax.set_ylim(0.5, 1.0)

plt.suptitle('Module 14 NB03: CNN for Sequence Classification', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cnn_sequence.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Visualize the learned convolutional filters as proper sequence logos (information
   content-weighted, as in NB13 Panel A). Do any filters resemble the planted motif?
2. Replace global max pooling with average pooling. Does performance change?
   Explain why in terms of the biological interpretation.
3. Add a second convolutional layer. Does performance improve on this dataset?
   What would a second-layer filter learn biologically (motif co-occurrence)?
4. Plot the precision-recall curve (from scratch, as in Module 13 NB09) and compare
   CNN vs. k-mer+LR. Which performs better at high recall?

---
## Step 10 — Quiz

1. Explain how a convolutional filter over a one-hot DNA sequence is analogous
   to a Position Weight Matrix.
2. What does global max pooling capture? Why is it preferred over average pooling
   for TF binding prediction?
3. What is the receptive field of a single convolutional layer with kernel size 11?
4. How would you extract the motif learned by a filter after training?
5. Why did DeepBind outperform PWM-based methods on ChIP-seq data?

---
## Step 12 — Reflection

> *[In one paragraph: explain the complete forward pass of the DeepBind CNN,
> from raw DNA sequence to binding probability, using the motif-detector analogy
> at each step. What biological decision does each layer correspond to?]*

---
*Next: `04_recurrent_networks_for_sequences.ipynb`*